# NWP Data Preprocessing — Cáceres Solar Forecast (TFT)

This notebook is a direct adaptation of `3_data_preprocessing.ipynb` for the NWP dataset
produced by `6_nwp_data_assembly.ipynb`. The pipeline is identical with three key differences:

## Differences from notebook 3

**1. Input file:** `caceres_nwp_dataset.csv` (23 columns: ERA5 + 5 forecast + 2 derived forecast + rest)

**2. Feature engineering (Step 5):** `kt_forecast` and `dewpoint_depression_C_forecast` were
already computed in notebook 6 from the raw forecast values. Only ERA5-derived features
(`kt`, `dewpoint_depression_C`) and `doy_sin`/`doy_cos` are computed here.

**3. Feature classification (Step 8 — the core architectural change):**

| Category | Columns | TFT role |
|---|---|---|
| Calendar + solar geometry | `hour_*`, `month_*`, `doy_*`, `solar_*`, `clearsky_ghi` | `known_future` |
| Forecast weather | `ssrd_wm2_forecast`, `temperature_2m_C_forecast`, `dewpoint_2m_C_forecast`, `surface_pressure_hPa_forecast`, `total_precip_mm_forecast`, `kt_forecast`, `dewpoint_depression_C_forecast` | `known_future` (decoder-visible) |
| ERA5 observed weather | `ssrd_wm2`, `strd_wm2`, `temperature_2m_C`, `dewpoint_2m_C`, `surface_pressure_hPa`, `total_precip_mm`, `kt`, `dewpoint_depression_C` | `observed` (encoder-only) |

Moving forecast columns to `known_future` allows the TFT decoder to attend to tomorrow's
weather signal when generating predictions — the key improvement over the standard model.

Note: `strd_wm2` has no forecast equivalent (Open-Meteo `terrestrial_radiation` is
top-of-atmosphere solar, not longwave thermal radiation). It stays encoder-only.

## Pipeline steps

| # | Step | Principle |
|---|---|---|
| 0 | Load, verify temporal continuity, rename target | Sanity check |
| 1 | Temporal train / val / test split | Split before any transforms |
| 2 | Impute missing target with 0 | Domain-knowledge correction (overnight hours) |
| 3 | Clip physical artifacts | Physics constraint (precip/irradiance ≥ 0) |
| 4 | Remove night-time target outliers | Unphysical measurement error |
| 5 | Feature engineering | kt, dewpoint depression, day-of-year cyclical (ERA5-based) |
| 6 | Drop redundant features | clearsky_dni, clearsky_dhi |
| 7 | Standardization (z-score) | Train-set mean/std applied to all splits |
| 8 | Feature classification for TFT | Known future (incl. forecast) vs. observed inputs |
| 9 | Save outputs | CSVs + nwp_preprocessing_params.json |

In [ ]:
import pandas as pd
import numpy as np
import json, os

DATA_DIR = os.path.join(os.getcwd(), 'data')
RAW_CSV  = os.path.join(DATA_DIR, 'caceres_nwp_dataset.csv')

## Step 0 — Load, verify temporal continuity, rename target

In [ ]:
df = pd.read_csv(RAW_CSV, parse_dates=['datetime_utc'], index_col='datetime_utc')

expected = pd.date_range('2023-01-01', '2025-12-31 23:00:00', freq='h')
assert len(df) == len(expected), f"Row count mismatch: {len(df)} vs {len(expected)}"
assert df.index.equals(expected), "Timestamp gaps detected"

df = df.rename(columns={'pv_generation_gwh': 'pv_generation_mwh'})
TARGET = 'pv_generation_mwh'

print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
print(f"Date range: {df.index[0]} → {df.index[-1]}")
print(f"Columns: {list(df.columns)}")
print(f"Missing target values: {df[TARGET].isna().sum():,} ({df[TARGET].isna().mean():.1%})")

## Step 1 — Temporal train / validation / test split

Identical splits to notebook 3 — same boundaries, same principle (split before any transforms).

| Set | Date Range |
|---|---|
| Train | 2023-01-01 → 2024-12-31 |
| Val   | 2025-01-01 → 2025-06-30 |
| Test  | 2025-07-01 → 2025-12-31 |

In [ ]:
TRAIN_END = '2024-12-31 23:00:00'
VAL_END   = '2025-06-30 23:00:00'

train = df.loc[:'2024-12-31 23:00:00'].copy()
val   = df.loc['2025-01-01':VAL_END].copy()
test  = df.loc['2025-07-01':].copy()

assert len(train) + len(val) + len(test) == len(df), "Split doesn't cover all rows"
assert train.index[-1] < val.index[0], "Train/val overlap"
assert val.index[-1] < test.index[0], "Val/test overlap"

splits = {'train': train, 'val': val, 'test': test}
for name, s in splits.items():
    pct = len(s) / len(df) * 100
    print(f"{name:5s}: {len(s):6,} rows ({pct:5.1f}%)  |  {s.index[0]} → {s.index[-1]}")

## Step 2 — Impute missing target with 0

Identical to notebook 3. 100% of missing target values occur overnight — ESIOS doesn't
report near-zero generation. Domain-knowledge fill, no leakage risk.

In [ ]:
for name, s in splits.items():
    n_before = s[TARGET].isna().sum()
    s[TARGET] = s[TARGET].fillna(0.0)
    print(f"{name:5s}: imputed {n_before:,} missing values → 0.0")

for name, s in splits.items():
    assert s[TARGET].isna().sum() == 0
print("\nNo NaN in target")

## Step 3 — Clip physical artifacts

Same as notebook 3, extended to forecast columns. ERA5 and Open-Meteo can both produce
tiny negative values for precipitation and irradiance due to floating-point arithmetic.
Both are physically non-negative.

In [ ]:
clip_cols = ['total_precip_mm', 'ssrd_wm2', 'total_precip_mm_forecast', 'ssrd_wm2_forecast']
for name, s in splits.items():
    for col in clip_cols:
        n_neg = (s[col] < 0).sum()
        s[col] = s[col].clip(lower=0.0)
        if n_neg > 0:
            print(f"{name:5s}: clipped {n_neg} negative values in {col}")

print("Physical clipping done")

## Step 4 — Remove night-time target outliers

Identical to notebook 3. Zeroes unphysical generation when solar zenith > 95°.

In [ ]:
NIGHT_ZENITH = 95
NIGHT_GEN_THRESHOLD = 0.1

for name, s in splits.items():
    night_outlier = (s['solar_zenith'] > NIGHT_ZENITH) & (s[TARGET] > NIGHT_GEN_THRESHOLD)
    n_outliers = night_outlier.sum()
    if n_outliers > 0:
        max_val = s.loc[night_outlier, TARGET].max()
        s.loc[night_outlier, TARGET] = 0.0
        print(f"{name:5s}: zeroed {n_outliers} night-time outliers (max was {max_val:.1f} MWh)")
    else:
        print(f"{name:5s}: no night-time outliers found")

## Step 5 — Feature engineering

`kt_forecast` and `dewpoint_depression_C_forecast` were already computed in notebook 6
from the raw forecast values before splitting. We derive the ERA5 equivalents here as in
notebook 3, plus `doy_sin`/`doy_cos` which apply to both.

| Feature | Source | Formula |
|---|---|---|
| `kt` | ERA5 | `ssrd_wm2 / clearsky_ghi`, clipped [0, 1.5] |
| `dewpoint_depression_C` | ERA5 | `temperature_2m_C − dewpoint_2m_C` |
| `doy_sin`, `doy_cos` | index | sin/cos of day-of-year / 365.25 |
| `kt_forecast` | already present | computed in notebook 6 |
| `dewpoint_depression_C_forecast` | already present | computed in notebook 6 |

In [ ]:
for name, s in splits.items():
    # kt from ERA5
    s['kt'] = np.where(
        s['clearsky_ghi'] > 1.0,
        s['ssrd_wm2'] / s['clearsky_ghi'],
        0.0
    )
    s['kt'] = s['kt'].clip(0.0, 1.5)

    # dewpoint depression from ERA5
    s['dewpoint_depression_C'] = s['temperature_2m_C'] - s['dewpoint_2m_C']

    # day-of-year cyclical
    doy = s.index.dayofyear
    s['doy_sin'] = np.sin(2 * np.pi * doy / 365.25)
    s['doy_cos'] = np.cos(2 * np.pi * doy / 365.25)

print(f"Feature engineering done. Train shape: {train.shape}")
print(f"kt_forecast already present: {'kt_forecast' in train.columns}")
print(f"dewpoint_depression_C_forecast already present: {'dewpoint_depression_C_forecast' in train.columns}")

## Step 6 — Drop redundant features

`clearsky_dni` and `clearsky_dhi` are highly collinear with `clearsky_ghi` (r > 0.93).
Same rationale as notebook 3.

In [ ]:
drop_cols = ['clearsky_dni', 'clearsky_dhi']
for name, s in splits.items():
    splits[name] = s.drop(columns=drop_cols)

train, val, test = splits['train'], splits['val'], splits['test']

print(f"Dropped: {drop_cols}")
print(f"Remaining columns ({train.shape[1]}):")
print(list(train.columns))

## Step 7 — Standardization (z-score)

Same approach as notebook 3: compute on training set only, apply to all splits.

Forecast columns use their **own** training-set stats (not ERA5 stats), since the
Open-Meteo and ERA5 distributions differ slightly. This gives better-calibrated
z-scores for each data source independently.

In [ ]:
# ERA5 features (same as notebook 3)
era5_features_to_scale = [
    'dewpoint_2m_C', 'temperature_2m_C', 'surface_pressure_hPa',
    'total_precip_mm', 'ssrd_wm2', 'strd_wm2',
    'solar_zenith', 'solar_azimuth', 'clearsky_ghi',
    'kt', 'dewpoint_depression_C',
]
# forecast features — scaled independently with their own train stats
forecast_features_to_scale = [
    'ssrd_wm2_forecast', 'temperature_2m_C_forecast', 'dewpoint_2m_C_forecast',
    'surface_pressure_hPa_forecast', 'total_precip_mm_forecast',
    'kt_forecast', 'dewpoint_depression_C_forecast',
]
features_to_scale = era5_features_to_scale + forecast_features_to_scale

cyclical_features = ['hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos']

# compute stats from training set ONLY
train_means = train[features_to_scale].mean()
train_stds  = train[features_to_scale].std()
target_mean = train[TARGET].mean()
target_std  = train[TARGET].std()

print("Training set statistics:")
print(pd.DataFrame({'mean': train_means, 'std': train_stds}).round(4))
print(f"\nTarget ({TARGET}): mean={target_mean:.4f}, std={target_std:.4f}")

In [ ]:
for name in splits:
    splits[name][features_to_scale] = (splits[name][features_to_scale] - train_means) / train_stds
    splits[name][TARGET] = (splits[name][TARGET] - target_mean) / target_std

train, val, test = splits['train'], splits['val'], splits['test']

train_check = train[features_to_scale].agg(['mean', 'std']).T
print("Post-standardization check (train set):")
print(train_check.round(6))
print(f"\nTarget — train mean: {train[TARGET].mean():.6f}, std: {train[TARGET].std():.6f}")

## Step 8 — Feature classification for TFT

This is the core architectural change vs. notebook 3.

The 7 forecast columns are added to `known_future` (decoder-visible). This means the TFT
decoder can attend to tomorrow's predicted weather when generating the 24h forecast —
the structural improvement the standard model (notebook 4) was missing.

ERA5 originals stay in `observed` (encoder-only), as the model still benefits from
knowing what actually happened in the past 7 days.

In [ ]:
known_future = [
    # calendar + solar geometry (deterministic, same as notebook 3)
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos',
    'solar_zenith', 'solar_azimuth', 'clearsky_ghi',
    # NWP forecast weather (NEW — decoder-visible)
    'ssrd_wm2_forecast', 'temperature_2m_C_forecast', 'dewpoint_2m_C_forecast',
    'surface_pressure_hPa_forecast', 'total_precip_mm_forecast',
    'kt_forecast', 'dewpoint_depression_C_forecast',
]
observed = [
    # ERA5 observed weather (encoder-only, same as notebook 3)
    'dewpoint_2m_C', 'temperature_2m_C', 'surface_pressure_hPa',
    'total_precip_mm', 'ssrd_wm2', 'strd_wm2',
    'kt', 'dewpoint_depression_C',
]
target_col = [TARGET]

all_classified = set(known_future + observed + target_col)
all_columns = set(train.columns)
assert all_classified == all_columns, (
    f"Unclassified: {all_columns - all_classified}  |  Extra: {all_classified - all_columns}"
)

print(f"Known future inputs ({len(known_future)}):")
for c in known_future:
    marker = '← NEW (forecast)' if '_forecast' in c else ''
    print(f"  {c}  {marker}")
print(f"\nObserved inputs ({len(observed)}): {observed}")
print(f"\nTarget: {target_col}")
print(f"\nTotal: {len(all_classified)} columns")

## Step 9 — Save outputs

In [ ]:
col_order = known_future + observed + target_col

for name in splits:
    out_path = os.path.join(DATA_DIR, f'nwp_{name}_processed.csv')
    splits[name][col_order].to_csv(out_path)
    print(f"Saved {out_path}  ({splits[name].shape[0]:,} rows x {splits[name].shape[1]} cols)")

params = {
    'split_boundaries': {
        'train': ['2023-01-01 00:00:00', TRAIN_END],
        'val':   ['2025-01-01 00:00:00', VAL_END],
        'test':  ['2025-07-01 00:00:00', '2025-12-31 23:00:00'],
    },
    'target_column': TARGET,
    'features_to_scale': features_to_scale,
    'cyclical_features': cyclical_features,
    'train_means': train_means.to_dict(),
    'train_stds': train_stds.to_dict(),
    'target_mean': float(target_mean),
    'target_std': float(target_std),
    'dropped_columns': drop_cols,
    'feature_classification': {
        'known_future': known_future,
        'observed': observed,
        'target': target_col,
    },
    'night_outlier_params': {
        'zenith_threshold': NIGHT_ZENITH,
        'generation_threshold_mwh': NIGHT_GEN_THRESHOLD,
    },
}

params_path = os.path.join(DATA_DIR, 'nwp_preprocessing_params.json')
with open(params_path, 'w') as f:
    json.dump(params, f, indent=2)
print(f"\nSaved {params_path}")
print("\nNext step: run 4b_nwp_training.ipynb")

## Verification

In [ ]:
print("=" * 70)
print("VERIFICATION CHECKS")
print("=" * 70)

# 1. no NaN
for name in splits:
    nans = splits[name].isna().sum().sum()
    assert nans == 0, f"{name} has {nans} NaN values"
print("[PASS] No NaN values in any split")

# 2. non-overlapping, full coverage
assert splits['train'].index[-1] < splits['val'].index[0]
assert splits['val'].index[-1] < splits['test'].index[0]
assert sum(len(s) for s in splits.values()) == 26_304
print("[PASS] Splits are contiguous, non-overlapping, cover full dataset")

# 3. standardization
for col in features_to_scale:
    m = splits['train'][col].mean()
    s = splits['train'][col].std()
    assert abs(m) < 1e-6, f"{col} mean = {m}"
    assert abs(s - 1.0) < 1e-6, f"{col} std = {s}"
assert abs(splits['train'][TARGET].mean()) < 1e-6
assert abs(splits['train'][TARGET].std() - 1.0) < 1e-6
print("[PASS] Standardized features have mean ~ 0, std ~ 1 on training set")

# 4. cyclical features untouched
for col in cyclical_features:
    assert splits['train'][col].min() >= -1.001
    assert splits['train'][col].max() <= 1.001
print("[PASS] Cyclical features remain in [-1, 1]")

# 5. forecast columns are in known_future
forecast_cols_check = [c for c in known_future if '_forecast' in c]
assert len(forecast_cols_check) == 7, f"Expected 7 forecast cols in known_future, got {len(forecast_cols_check)}"
print(f"[PASS] {len(forecast_cols_check)} forecast columns classified as known_future (decoder-visible)")

print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
for name in ['train', 'val', 'test']:
    s = splits[name]
    print(f"{name:5s}: {s.shape[0]:6,} rows x {s.shape[1]} cols  |  "
          f"{s.index[0].strftime('%Y-%m-%d')} -> {s.index[-1].strftime('%Y-%m-%d')}")
print(f"\nFeatures: {len(known_future)} known future ({len(forecast_cols_check)} forecast) "
      f"+ {len(observed)} observed + 1 target = {len(all_classified)} total")
print("All checks passed")